In [ ]:
!pip install groq python-dotenv

In [2]:
import os
from groq import Groq
from getpass import getpass

# Ask user for API Key (hidden input)
api_key = getpass("Paste your GROQ API Key here: ")

# Create Groq client
client = Groq(api_key=api_key)

print("Groq Client Initialized Successfully!")

Groq Client Initialized Successfully!


In [9]:
MODEL_NAME = "llama-3.3-70b-versatile"

MODEL_CONFIG = {

    "technical": {
        "system_prompt": """
You are a Senior Software Engineer and debugging specialist.
You provide precise, code-focused, step-by-step technical solutions.
If possible, include corrected code snippets.
Avoid casual language.
"""
    },

    "billing": {
        "system_prompt": """
You are a customer billing support specialist.
You are polite, empathetic, and professional.
Explain billing issues clearly, discuss refund policies, and guide users step-by-step.
Always acknowledge the user's concern emotionally first.
"""
    },

    "general": {
        "system_prompt": """
You are a friendly AI assistant for general conversations and casual questions.
Keep responses helpful and natural.
"""
    },

    # BONUS TOOL EXPERT
    "tool": {
        "system_prompt": """
You are a system assistant that uses external tools to fetch real-world data instead of guessing.
If the router sends a request here, a tool will handle it.
"""
    }
}

In [10]:
def call_llm(system_prompt, user_prompt, temperature=0.7):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [11]:
def route_prompt(user_input):

    routing_prompt = f"""
Classify the following user query into ONE of these categories:

technical → programming, bugs, errors, code, debugging
billing → payments, refunds, charges, subscription, invoices
tool → requests for real-time data like crypto price, weather, stock price
general → casual conversation or anything else

Return ONLY ONE WORD from this list:
technical
billing
tool
general

User Query:
{user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,  # VERY IMPORTANT for consistency
        messages=[
            {"role": "system", "content": "You are an intent classification engine."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()

    return category

In [22]:
def get_bitcoin_price():
    """
    Simulates fetching Bitcoin price from an external API.
    In a real implementation, this would call a crypto API like CoinGecko or Binance.
    """
    # Simulated response - in production, you'd use requests library to call an actual API
    return "Bitcoin (BTC) current price: $52,341.27 USD (Note: This is a simulated price. For real-time data, integrate with a crypto API like CoinGecko.)"

In [17]:
def process_request(user_input):

    # Step 1: Route
    category = route_prompt(user_input)
    print("Routed to:", category)

    # Step 2: Tool Handling
    if category == "tool":
        return get_bitcoin_price()

    # Step 3: Select Expert
    if category not in MODEL_CONFIG:
        category = "general"

    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 4: Call Expert
    final_answer = call_llm(system_prompt, user_input, temperature=0.7)

    return final_answer

# Technical 

In [18]:
query = "My python script throws IndexError on line 5 while accessing a list."

print(process_request(query))

Routed to: technical
**IndexError Exception in Python**

The `IndexError` exception in Python occurs when you attempt to access an index in a sequence (such as a list or a string) that does not exist.

### Step-by-Step Debugging

To resolve this issue, follow these steps:

1. **Review the code**: Examine the line of code where the exception is occurring (line 5 in this case).
2. **Check the list index**: Ensure that the index you are trying to access is within the bounds of the list.
3. **Verify list initialization**: Confirm that the list is properly initialized and populated with data before attempting to access its elements.

### Example Code with Error

```python
# Example list
my_list = [1, 2, 3]

# Attempt to access an index that does not exist
print(my_list[5])  # This will throw an IndexError
```

### Corrected Code

To fix the issue, you can add a check to ensure that the index is within the bounds of the list:

```python
# Example list
my_list = [1, 2, 3]

# Define the index 

# Billing Expert

In [19]:
query = "I was charged twice for my subscription. I need a refund."

print(process_request(query))

Routed to: billing
I'm so sorry to hear that you've been charged twice for your subscription. I can imagine how frustrating that must be for you, and I'm here to help resolve the issue as quickly as possible.

First, please know that I'm committed to making things right, and I'll do my best to ensure that you receive a refund for the duplicate charge. Can you please provide me with more details about the issue, such as the date of the duplicate charge and the amount that was taken?

Additionally, I'll need to verify some information to process the refund. Could you please confirm your subscription details, including your account name and the type of subscription you have? This will help me to locate the issue and expedite the refund process.

Once I have this information, I'll guide you through the next steps and ensure that the refund is processed promptly. Your satisfaction is my top priority, and I appreciate your patience and cooperation in resolving this matter.


# General expert

In [20]:
query = "Tell me a joke about programmers"

print(process_request(query))

Routed to: general
Here's one: Why do programmers prefer dark mode?

Because light attracts bugs. (get it?)


# Tool expert

In [23]:
query = "What is the current price of Bitcoin?"

print(process_request(query))

Routed to: tool
Bitcoin (BTC) current price: $52,341.27 USD (Note: This is a simulated price. For real-time data, integrate with a crypto API like CoinGecko.)
